# 🔥 Chapter 3 — Tensor Operations & Mathematical Functions

---

### 🎯 What you'll learn
- Element-wise and matrix operations
- Broadcasting rules and corner cases
- Reduction operations (sum, mean, max, argmax)
- In-place operations and why they're dangerous with autograd

## 3.1 — Element-wise Operations

In [ ]:
import torch

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"Add:      {a + b}")       # or torch.add(a, b)
print(f"Subtract: {a - b}")
print(f"Multiply: {a * b}")       # Element-wise, NOT matrix multiply
print(f"Divide:   {a / b}")
print(f"Power:    {a ** 2}")
print(f"Sqrt:     {torch.sqrt(a)}")
print(f"Abs:      {torch.abs(torch.tensor([-1., 2., -3.]))}")
print(f"Clamp:    {torch.clamp(a, min=1.5, max=2.5)}")  # Clip values

## 3.2 — Matrix Operations

In [ ]:
A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[5., 6.], [7., 8.]])

# Matrix multiplication — 3 equivalent ways
mm = torch.mm(A, B)            # Only for 2D
matmul = torch.matmul(A, B)    # Works for any dims (broadcasting)
at_sign = A @ B                 # Shorthand for matmul

print(f"A @ B:\n{at_sign}")

# ⚠️ Corner Case: * vs @
print(f"\nA * B (element-wise):\n{A * B}")
print(f"A @ B (matrix mult):\n{A @ B}")
# These are COMPLETELY different! Common bug source.

# Dot product (1D only)
v1 = torch.tensor([1., 2., 3.])
v2 = torch.tensor([4., 5., 6.])
print(f"\nDot product: {torch.dot(v1, v2)}")

# Batch matrix multiplication
batch_A = torch.randn(10, 3, 4)  # 10 matrices of 3x4
batch_B = torch.randn(10, 4, 5)  # 10 matrices of 4x5
result = torch.bmm(batch_A, batch_B)  # → (10, 3, 5)
print(f"Batch matmul shape: {result.shape}")

## 3.3 — ⚠️ Broadcasting Rules

Broadcasting lets tensors of different shapes work together. But it's also a **major source of silent bugs**.

### Rules (from trailing dimension):
1. Dimensions are compared **right to left**.
2. Dimensions must be **equal**, or one of them must be **1**.
3. A dimension of **1** is "stretched" to match the other.

In [ ]:
# ✅ Valid broadcasting
a = torch.ones(3, 4)       # (3, 4)
b = torch.ones(4)          # (4,)  →  broadcast to (3, 4)
print(f"(3,4) + (4,): {(a + b).shape}")

c = torch.ones(3, 1)       # (3, 1) → broadcast to (3, 4)
print(f"(3,4) + (3,1): {(a + c).shape}")

# ⚠️ Corner Case: Silent shape mismatch!
x = torch.tensor([[1.], [2.], [3.]])   # (3, 1)
y = torch.tensor([10., 20., 30., 40.]) # (4,)
result = x + y  # (3, 1) + (4,) → broadcasts to (3, 4)!
print(f"\n⚠️ (3,1) + (4,) = {result.shape}")
print(result)
# This might NOT be what you intended! You might have wanted (3,) + (3,)

# ❌ Invalid broadcasting
try:
    bad = torch.ones(3, 4) + torch.ones(2, 4)  # 3 ≠ 2, neither is 1
except RuntimeError as e:
    print(f"\n❌ Error: {e}")

## 3.4 — Reduction Operations

In [ ]:
t = torch.tensor([[1., 2., 3.],
                     [4., 5., 6.]])

print(f"Sum all:      {t.sum()}")
print(f"Sum cols:     {t.sum(dim=0)}")     # Sum down rows → (3,)
print(f"Sum rows:     {t.sum(dim=1)}")     # Sum across cols → (2,)

print(f"Mean:         {t.mean()}")
print(f"Max:          {t.max()}")
print(f"Argmax (flat):{t.argmax()}")        # Index in flattened tensor
print(f"Argmax dim=1: {t.argmax(dim=1)}")   # Index of max per row

# ⚠️ Corner Case: keepdim
sum_no_keep = t.sum(dim=1)                # shape: (2,)
sum_keepdim = t.sum(dim=1, keepdim=True)  # shape: (2, 1)
print(f"\nWithout keepdim: {sum_no_keep.shape}")
print(f"With keepdim:    {sum_keepdim.shape}")
# keepdim=True is essential when you want to broadcast the result
# back against the original tensor (e.g., normalization)

# Normalization example
normalized = t / t.sum(dim=1, keepdim=True)
print(f"\nNormalized rows:\n{normalized}")
print(f"Row sums check: {normalized.sum(dim=1)}")  # Should be [1, 1]

## 3.5 — ⚠️ In-place Operations and Autograd

In [ ]:
# In-place operations end with _ (underscore)
x = torch.tensor([1., 2., 3.])
x.add_(10)       # In-place add
x.mul_(2)        # In-place multiply
print(f"After in-place: {x}")

# ⚠️ DANGER: In-place ops break autograd!
w = torch.tensor([1., 2., 3.], requires_grad=True)

# This would cause an error during backward():
# w.add_(1)  # ❌ RuntimeError: in-place operation on a leaf Variable

# ✅ Use out-of-place operations with autograd
w_new = w + 1   # Creates new tensor, gradient graph intact
print(f"Safe: {w_new}")

## ✏️ Exercises

In [ ]:
# Exercise 1: Given a (5, 3) matrix of exam scores (5 students, 3 exams),
# compute each student's average score using reduction ops.
scores = torch.randint(50, 100, (5, 3), dtype=torch.float32)
# YOUR CODE HERE


# Exercise 2: Manually normalize a tensor to have zero mean and unit std deviation
# (z = (x - mean) / std). Use keepdim properly.
data = torch.randn(4, 5)
# YOUR CODE HERE


# Exercise 3: What shapes will these broadcast to? Predict first, then verify:
# (5, 3, 1) + (1, 3, 4) = ?
# (2, 1, 6) + (1, 5, 1) = ?
# YOUR CODE HERE
